In [4]:
from transformers import AutoModelForCausalLM
from anticipation import ops
from anticipation.convert import midi_to_events, events_to_midi
from anticipation.tokenize import extract_instruments
from anticipation.sample import generate
import torch

In [5]:
MODELS = {
    "small":  "stanford-crfm/music-small-800k",
    "medium": "stanford-crfm/music-medium-800k",
    "large":  "stanford-crfm/music-large-800k",
}

def load(size="small", device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(MODELS[size], torch_dtype=dtype, use_safetensors=False).to(device).eval()
    return model

def continue_midi(model, midi_path, out_path, prompt_seconds=5, generate_seconds=20, top_p=0.98):
    """Feed a MIDI file as a prompt, generate a continuation, save result."""
    events = midi_to_events(str(midi_path))
    history = ops.clip(events, 0, prompt_seconds, clip_duration=False)
    new_events = generate(model, prompt_seconds, prompt_seconds + generate_seconds,
                          inputs=history, top_p=top_p)
    events_to_midi(new_events).save(out_path)

def accompany_melody(model, midi_path, out_path, melody_program=53,
                    prompt_seconds=5, generate_seconds=20, top_p=0.98):
    """Extract a melody track, generate accompaniment around it, save combined MIDI."""
    events = midi_to_events(str(midi_path))
    events, melody = extract_instruments(events, [melody_program])
    history = ops.clip(events, 0, prompt_seconds, clip_duration=False)
    accompaniment = generate(model, prompt_seconds, prompt_seconds + generate_seconds,
                             inputs=history, controls=melody, top_p=top_p)
    combined = ops.combine(accompaniment, melody)
    events_to_midi(combined).save(out_path)

In [ ]:
# Stanford CRFM music model playground

from pathlib import Path
from IPython.display import FileLink, Markdown, display

# Knobs to play with. Start small on CPU; bump these once the flow feels clear.
MODEL_SIZE = "small"      # "small", "medium", or "large"
TOP_P = 0.95              # lower = safer/more predictable, higher = more surprising
PROMPT_SECONDS = 15        # how much of the input MIDI the model hears first
GENERATE_SECONDS = 10     # how many seconds of new music to ask for

OUTPUT_DIR = Path("generated_music")
OUTPUT_DIR.mkdir(exist_ok=True)

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

def summarize_events(events, label="events"):
    """Small sanity check for anticipation event tokens."""
    instruments = ops.get_instruments(events)
    top_instruments = sorted(instruments.items(), key=lambda item: item[1], reverse=True)[:8]
    print(f"{label}: {len(events) // 3:,} note-like events, ~{ops.max_time(events):.1f}s long")
    print("Top MIDI programs:", top_instruments if top_instruments else "none")

# Pick a default MIDI prompt from the repo. Replace this with any .mid/.midi path you want.
prompt_midi = first_existing(
    "../data/clean_midi/Bob Dylan/Blowin' in the Wind.mid",
    "data/clean_midi/Bob Dylan/Blowin' in the Wind.mid",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = globals().get("model")
if model is None:
    print(f"Loading {MODEL_SIZE!r} model on {device}. The first run may download weights.")
    model = load(MODEL_SIZE, device=device)
else:
    print(f"Reusing existing model: {model.__class__.__name__}")

config_bits = {
    name: getattr(model.config, name)
    for name in ["model_type", "vocab_size", "n_layer", "num_hidden_layers", "n_head", "num_attention_heads", "n_embd", "hidden_size"]
    if hasattr(model.config, name)
}
display(Markdown("### Model quick look"))
print(model.__class__.__name__)
print(config_bits)

# Experiment 1: generate from silence / no prompt, without context
# This shows the model's unconditional prior: what it tends to write with no MIDI context.
with torch.inference_mode():
    blank_events = generate(model, 0, GENERATE_SECONDS, top_p=TOP_P)

blank_out = OUTPUT_DIR / f"crfm_{MODEL_SIZE}_blank_top-p-{TOP_P}.mid"
events_to_midi(blank_events).save(blank_out)
summarize_events(blank_events, "Blank generation")
display(FileLink(str(blank_out)))

# Experiment 2: feed the model the first few seconds of an existing MIDI and ask it to continue.
if prompt_midi is None:
    print("No default prompt MIDI found. Set `prompt_midi = Path('your/file.mid')` and rerun this part.")
else:
    print(f"Prompt MIDI: {prompt_midi}")
    original_events = midi_to_events(str(prompt_midi))
    prompt_events = ops.clip(original_events, 0, PROMPT_SECONDS, clip_duration=False)
    summarize_events(prompt_events, "Prompt slice")

    with torch.inference_mode():
        continued_events = generate(
            model,
            PROMPT_SECONDS,
            PROMPT_SECONDS + GENERATE_SECONDS,
            inputs=prompt_events,
            top_p=TOP_P,
        )

    continued_out = OUTPUT_DIR / f"crfm_{MODEL_SIZE}_continuation_top-p-{TOP_P}.mid"
    events_to_midi(continued_events).save(continued_out)
    summarize_events(continued_events, "Prompt + continuation")
    display(FileLink(str(continued_out)))

# can tweak the hyperparamters up there

Reusing existing model: GPT2LMHeadModel


### Model quick look

GPT2LMHeadModel
{'model_type': 'gpt2', 'vocab_size': 55028, 'n_layer': 12, 'num_hidden_layers': 12, 'n_head': 12, 'num_attention_heads': 12, 'n_embd': 768, 'hidden_size': 768}


 99%|█████████▉| 992/1000 [10:14<00:04,  1.61it/s]

Blank generation: 438 note-like events, ~9.9s long
Top MIDI programs: [(128, 195), (25, 78), (52, 44), (28, 30), (35, 29), (0, 21), (69, 17), (73, 16)]


/Users/pr3s10/Desktop/InteractiveMusicAnalyzer/notebooks/generated_music/crfm_small_blank_top-p-0.95.mid

Prompt MIDI: ../data/clean_midi/Bob Dylan/Blowin' in the Wind.mid
Prompt slice: 0 note-like events, ~0.0s long
Top MIDI programs: none


1071it [06:56,  2.57it/s]                          

Prompt + continuation: 347 note-like events, ~14.7s long
Top MIDI programs: [(25, 284), (128, 63)]


/Users/pr3s10/Desktop/InteractiveMusicAnalyzer/notebooks/generated_music/crfm_small_continuation_top-p-0.95.mid

### Note: Packaged and uses hugging face GPT-2-style causal transformer architecture to also help predict the next music note, similar to regular LM predicting next word token. So, Stanford CRFM msuic modle uses GPT-2 langauge model with a custom music-token vocabulary instead of text. It then converts those predicted event tokens back into MIDI.